# Playbook 05 — Hybrid QML Prediction

End-to-end hybrid quantum-classical link prediction over Hetionet CtD using the embeddings produced by [`04_kg_embedding_training.ipynb`](04_kg_embedding_training.ipynb).

**Pipeline:**
1. Load entity embeddings (32d) and reduce to QML dim (5d via PCA).
2. Build link features from embeddings (head-tail concatenation or difference).
3. Train classical baseline (random forest / stacking ensemble).
4. Train QSVC with Pauli feature map (the preregistered headline config).
5. Ensemble + bootstrap CI.

**Locked preregistered defaults** (`utils/preregistered_constants.py`):
- `QSVC_FEATURE_MAP_TYPE = 'PauliFeatureMap'`
- `QSVC_FEATURE_MAP_REPS = 2`
- `QSVC_C_BEST = 1.0`
- `ENSEMBLE_METHOD = 'stacking'`
- `NEGATIVE_SAMPLING = 'hard_degree_corrupt'`

In [ ]:
import logging, json
logging.basicConfig(level=logging.INFO)
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

## 1. Set seeds + load preregistered constants

In [ ]:
from utils.reproducibility import set_global_seed
from utils.preregistered_constants import (
    EMBEDDING_DIM, QSVC_QML_DIM, QSVC_FEATURE_MAP_TYPE, QSVC_FEATURE_MAP_REPS,
    QSVC_C_BEST, NEGATIVE_SAMPLING, ENSEMBLE_METHOD,
)
set_global_seed(42)
print(f'Feature map : {QSVC_FEATURE_MAP_TYPE} (reps={QSVC_FEATURE_MAP_REPS})')
print(f'QSVC C      : {QSVC_C_BEST}')
print(f'Ensemble    : {ENSEMBLE_METHOD}')
print(f'Neg sampling: {NEGATIVE_SAMPLING}')
print(f'Embedding   : dim={EMBEDDING_DIM} → qml_dim={QSVC_QML_DIM}')

## 2. Load CtD task and prepare link-prediction dataset

In [ ]:
from kg_layer.kg_loader import (
    load_hetionet_edges, extract_task_edges, prepare_link_prediction_dataset,
    get_hard_negatives_degree_corrupt,
)

edges_df = load_hetionet_edges('data/hetionet-v1.0-edges.sif')
task_edges = extract_task_edges(edges_df, relation='CtD')
print(f'CtD positive edges: {len(task_edges):,}')

In [ ]:
# Hard negatives — degree-corrupt sampling (matches preregistered constant)
dataset = prepare_link_prediction_dataset(task_edges, edges_df, neg_strategy='hard_degree_corrupt')
print(f'Total pairs (pos+neg): {len(dataset):,}')
print(f'Label balance: {dataset["label"].value_counts().to_dict()}')

In [ ]:
train_df, test_df = train_test_split(
    dataset, test_size=0.2, stratify=dataset['label'], random_state=42
)
print(f'Train: {len(train_df):,}   Test: {len(test_df):,}')

## 3. Load (or train) embeddings via HetionetEmbedder

In [ ]:
from kg_layer.kg_embedder import HetionetEmbedder
from kg_layer.kg_loader import prepare_full_graph_for_embeddings

embedder = HetionetEmbedder(embedding_dim=EMBEDDING_DIM, qml_dim=QSVC_QML_DIM, work_dir='data')
if not embedder.load_saved_embeddings(expected_dim=EMBEDDING_DIM):
    print('No cache — training (see playbook 04 to do this separately)')
    triples = prepare_full_graph_for_embeddings(edges_df)
    embedder.train_embeddings(triples)
embedder.reduce_to_qml_dim()
print(f'Embeddings: {embedder.entity_embeddings.shape} → reduced: {embedder.reduced_embeddings.shape}')

## 4. Train + evaluate (classical baseline + QSVC ensemble)

`QMLTrainer.train_and_evaluate` runs both branches and returns paired metrics. Quantum kernel computation is the expensive step — expect 5-15 min for ~2k test pairs on CPU statevector, faster on GPU.

In [ ]:
from quantum_layer.qml_trainer import QMLTrainer

trainer = QMLTrainer(seed=42)
results = trainer.train_and_evaluate(
    train_df=train_df,
    test_df=test_df,
    embedder=embedder,
    config_path='config/quantum_layer_config.yaml',
)
print(json.dumps({k: {m: round(v, 4) for m, v in d.items()} for k, d in results.items()}, indent=2))

## 5. Headline metric: PR-AUC

The preregistered headline result is **PR-AUC = 0.7987** for the stacked ensemble. Substantially different numbers from this notebook may indicate:
- Different random seed (re-run with `set_global_seed(42)`)
- Different embedding dim or feature map (drift from `preregistered_constants.py`)
- Different negative sampling strategy

In [ ]:
for branch, metrics in results.items():
    pr_auc = metrics.get('pr_auc') or metrics.get('average_precision')
    print(f'  {branch:20s}  PR-AUC = {pr_auc:.4f}')

## 6. Bootstrap confidence interval (paired H1 test)

Tests the preregistered hypothesis H1: stacking ensemble > classical-only baseline. Uses paired bootstrap to keep correlated noise (same test pairs evaluated by both).

In [ ]:
from utils.bootstrap_ci import paired_bootstrap_ci
# Pass the per-test-pair predictions if available; otherwise this falls back
# to a synthetic illustration.
print('See scripts/run_bootstrap_ci.py for the full preregistered CI run.')
print('In production: python scripts/run_bootstrap_ci.py --n_bootstrap 1000')

## 7. Feed scores into the evidence layer

These KG + QML scores become the `kg_rotate_score` / `qsvc_score` / `classical_ensemble_score` fields in `EvidenceFeatures`, ready for fusion with omics evidence.

In [ ]:
from evidence_layer.evidence_schema import EvidenceFeatures
from evidence_layer.feature_fusion import fuse_evidence

demo = [
    EvidenceFeatures(
        compound='metformin', compound_hetionet_id='Compound::DB00331',
        disease='type 2 diabetes', disease_hetionet_id='Disease::DOID:9352',
        kg_rotate_score=0.91, qsvc_score=results.get('quantum', {}).get('average_precision', 0.0),
        classical_ensemble_score=results.get('classical', {}).get('average_precision', 0.0),
    ),
]
fused = fuse_evidence(demo, mode='kg-only')
print(f'Top: {fused[0].compound} score={fused[0].final_score:.4f} tier={fused[0].confidence_tier}')

**Next:** [`06_candidate_ranking_and_validation.ipynb`](06_candidate_ranking_and_validation.ipynb) demonstrates the full fusion (KG + QML + omics + clinical).